# Bass-LV in a few lines

The smallest working version of the construction in Conze and Henry-Labordère (2021,
[SSRN 3853085](https://ssrn.com/abstract=3853085)): the fixed point, the reset between
market dates and a path-dependent price, on the same SPX snapshot as the main notebook
(`bass_lv_replication.ipynb`). To keep it short, each marginal is a mean-one lognormal law
with the market's ATM volatility for that expiry, so the smile is ignored here; the main
notebook fits the full smile.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

import bass_lv as bl
from bass_lv.data import OTM_FILE, load_otm_quotes

term, _, spot = load_otm_quotes(Path("..") / "data" / OTM_FILE)  # see the README for the data
T = term["T"].to_numpy()
marginals = [bl.lognormal_marginal(row.atm_iv**2 * row["T"]) for _, row in term.iterrows()]
term[["T", "forward", "atm_iv"]].round(4)

,T,forward,atm_iv
expiry,,,
2019-09-20,0.0986,2849.5462,0.1954
2019-12-20,0.3479,2849.4489,0.1854
2020-06-19,0.8466,2847.3013,0.1775


## The iteration

For consecutive marginals $\mu_1,\mu_2$ and $\Delta=T_2-T_1$, let $F$ be the CDF of the latent
Brownian state at $T_1$:

```text
initialize F
repeat
    pre      = K_Delta * F                 # law of W just before T2
    terminal = F_mu2^{-1} o pre            # g: sends W_{T2-} to mu2
    start    = K_Delta * terminal          # h = f(T1, .), by the heat equation
    F_new    = F_mu1 o start               # A F, Theorem 2.1 Eq. (2)
    shift F_new so that F_new(0) = 1/2     # Remark 2.3
until sup |F_new - F| < tolerance
```

Written out with the package's heat kernel, starting from the Gaussian law of $W_{T_1}$:

In [2]:
w = bl.W_GRID
mu1, mu2, dt = marginals[0], marginals[1], T[1] - T[0]

F = norm.cdf(w / np.sqrt(T[0]))
for iteration in range(1, 1001):
    pre = bl.heat(F, w, dt, cdf=True)
    terminal = mu2.ppf(pre)
    start = bl.heat(terminal, w, dt)
    new = np.maximum.accumulate(bl.recenter(mu1.cdf(start)))
    step, F = np.max(np.abs(new - F)), new
    if step < 1e-5:
        break
print(f"converged in {iteration} iterations, last step {step:.1e}")

packaged = bl.calibrate_pair(mu1, mu2, T[0], T[1], initial="gaussian")
print("same fixed point as bl.calibrate_pair:", np.array_equal(F, packaged.latent_cdf))

converged in 8 iterations, last step 3.5e-06
same fixed point as bl.calibrate_pair: True


## From intervals to paths

`bl.calibrate` builds the first interval with the classical Bass map
$f(T_1,w)=F_{\mu_1}^{-1}(N(w/\sqrt{T_1}))$ and solves one fixed point per later interval. A
path needs one Gaussian draw per interval; at each market date the asset value is kept and the
latent state is reset through the next interval's start map (Eq. 8).

In [3]:
model = bl.calibrate(marginals, T, initial="gaussian")
X = bl.simulate_market_dates(model, n=50_000, seed=7)  # S_{T_i} / F_i

check = pd.DataFrame({
    "target mean": [m.mean() for m in marginals],
    "MC mean": X.mean(axis=0),
    "target ATM call": [m.call(1.0) for m in marginals],
    "MC ATM call": np.maximum(X - 1, 0).mean(axis=0),
}, index=[f"{e:%b %Y}" for e in term.index])
display(check.round(5))

price, se = bl.asian_call(X, term.forward.to_numpy(), term.discount.iloc[-1], spot)
print(f"Asian call, strike {spot:.2f}: {price:.2f} ± {se:.2f} index points (MC s.e.)")

,target mean,MC mean,target ATM call,MC ATM call
Sep 2019,1.0,0.99967,0.02449,0.02422
Dec 2019,1.0,0.99993,0.04362,0.04350
Jun 2020,1.0,1.00046,0.06508,0.06554


Asian call, strike 2847.60: 106.91 ± 0.75 index points (MC s.e.)


The payoff is $\big(\tfrac13(S_{T_1}+S_{T_2}+S_{T_3})-K\big)^+$ with $K$ the spot. It depends on
the joint law of the three fixings, which the marginals alone do not determine; Bass-LV supplies
that joint law. The standard error measures Monte Carlo noise only, not smile, dividend or model
risk. With 60,000 paths and the Eq. (26) initializer, the main notebook prices the same Asian at
106.53 ± 0.68; the gap to the price above is within Monte Carlo error.